In [2]:
import pprint
import evaluate
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

In [3]:
ds = load_dataset(
    'bitext/Bitext-customer-support-llm-chatbot-training-dataset',
    split="train")

In [4]:
first_thousand_points = ds[:1000]
train_ds = Dataset.from_dict(first_thousand_points)
 
evaluation_dataset = Dataset.from_dict(ds[1000:1020])

In [5]:
def merge_example(row):
    row['conversation'] = f"Query: {row['instruction']}\nResponse: {row['response']}"
    return row
 
train_ds = train_ds.map(merge_example)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 12123.74 examples/s]


In [6]:
print(train_ds[0]['conversation'])

Query: question about cancelling order {{Order Number}}
Response: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [7]:
model_name = "Maykeye/TinyLLama-v0"
 
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [8]:
from peft import LoraConfig

In [9]:
lora_config = LoraConfig(
    r=12,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['q_proj', 'v_proj']
)

In [10]:
training_arguments_lora = TrainingArguments(
    output_dir="./tiny_lora",
    per_device_train_batch_size=1,
    learning_rate=2e-3,
    max_grad_norm=0.3,
    max_steps=200,
    gradient_accumulation_steps=2,
    save_steps=10,
    logging_steps=10,
    use_cpu=True,
    report_to="none",
    seed=42,
)

In [11]:
trainer_lora = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field='conversation',
    max_seq_length=250,
    args=training_arguments_lora,
    peft_config=lora_config
)


Map: 100%|██████████| 1000/1000 [00:00<00:00, 4310.09 examples/s]
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:318: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [12]:
trainer_lora.model.print_trainable_parameters()

trainable params: 24,576 || all params: 4,645,952 || trainable%: 0.5290


In [13]:
trainer_lora.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,8.030300
20,7.295800
30,6.899600
40,6.818800
50,6.648000
60,6.425600
70,6.345300
80,6.206900
90,6.120000
100,6.088200


TrainOutput(global_step=200, training_loss=6.27275915145874, metrics={'train_runtime': 63.8746, 'train_samples_per_second': 6.262, 'train_steps_per_second': 3.131, 'total_flos': 1455409081728.0, 'train_loss': 6.27275915145874, 'epoch': 0.4})

In [15]:
def generate_predictions_and_reference(dataset):
    predictions = []
    references = []
    for row in dataset:
        prompt = f"Query: {row['instruction']}\nResponse:"
        inputs = tokenizer.encode(prompt, return_tensors="pt")
        outputs = model.generate(
            inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
        decoded_outputs = tokenizer.decode(
            outputs[0, inputs.shape[1]:], skip_special_tokens=True)
        references += [row["response"]]
        predictions += [decoded_outputs]
    return references, predictions

In [16]:
rouge = evaluate.load('rouge')

In [18]:
model = trainer_lora.model
references, predictions = generate_predictions_and_reference(evaluation_dataset)
results_lora = rouge.compute(predictions=predictions, references=references)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [21]:
results_after = rouge.compute(predictions=predictions, references=references)
print("LoRA:", {k: round(float(v), 3) for k, v in results_after.items()})

print("\nSample output (LoRA):")
print("Query:   ", evaluation_dataset[0]["instruction"])
print("Model:   ", predictions[0])
print("Expected:", references[0])

LoRA: {'rouge1': 0.258, 'rouge2': 0.037, 'rougeL': 0.184, 'rougeLsum': 0.194}

Sample output (LoRA):
Query:    want help adding an item to order {{Order Number}}
Model:    I'm going to go to the store to buy the help of the story "No, I'm here to help you find the way this can find the most matter,"
3 with this special adventure, you're going to find the
Expected: Thank you for getting in touch to us for assistance with adding an item to your order. We understand the importance of getting your order just right. To help you with this, could you please provide us with the details of the item you would like to add? By having this information, we can ensure that your order is complete and meets your expectations. We appreciate your cooperation and look forward to assisting you further.
